# Extra X4 · An end-to-end voice assistant

**Time:** 45 to 60 min · **Work in teams of 3** · **Optional lab: use it only if there is time**

**Where we are in the day:** in Lab 21 the assistant listened to a few clips and acted. Extra lab X3 gave us a full dialogue manager for a TV, with context, confirmations and repair. Now we join everything and let the assistant **answer in audio**.
**Today's question:** what does the whole chain, from the user's voice to the assistant's voice, cost in time, and what breaks when the room is noisy?
**What you will do:**
- Build the chain: speech to text (Whisper), the dialogue manager, text to speech, so that the assistant speaks its answer.
- Run whole conversations from audio clips and draw a timing waterfall for every turn.
- Add noise at three levels, and measure how the heard text, the intent and the final result change.
- List where the chain breaks and what you would do about each break, then add a "did you say ...?" check.

**What you will see:** the four stages of one turn with their milliseconds, a table of what noise does to the text and to the result, and a count of the turns where a speech mistake did not matter and the turns where it did. In Lab 21 speech recognition took 78 to 343 ms per clip and a mistake such as "Hose the living room blinds" still closed the blinds; here we check how far that luck goes.
**Why it matters on a device:** a user does not care how good your model is on text. They judge the whole chain: how long they wait, and whether it works in a kitchen with a fan on.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- The cells take about **3 minutes** in all, once the models are on the disk. No microphone is needed.
- If a cell takes longer than that, wait until the time is over, then interrupt the kernel and ask the trainer. The first run downloads Whisper small and a small text-to-speech model (together a little over 1 GB), which can take a few minutes.
- Before you start: extra lab X3 should have been run, so that the TV adapter exists. If it does not, the lab says so in the first cell and uses the home assistant of Day 2 instead; everything else works in the same way.
- **Recorded clips:** this course comes with spoken clips in the folders `data/voice/tv` and `data/voice/home`, named `conv1_turn1.wav`, `conv1_turn2.wav`, `conv2_turn1.wav` and so on (conversation number, turn number). The lab uses a clip when it finds one, and makes a robot-voice clip with the same name when it does not. You can put your own recordings there too. The made clips are saved in `outputs/x_voice`, so you can see which sentences to record.

**Words used in this lab:**
- **Speech recognition:** turning a recording into text. **Whisper** is the speech model we use.
- **Text to speech:** turning text into a recording. We use the small model `facebook/mms-tts-eng`.
- **Waterfall:** a table of how many milliseconds each stage of one turn took, in order.
- **Signal-to-noise ratio (dB):** how much stronger the voice is than the noise. 20 dB is a light noise, 12 dB a medium one, 6 dB a heavy one, 0 dB means the noise is as strong as the voice.
- **Word error rate:** of the words the user said, the share that the recogniser got wrong (missing, extra or changed).
- **Wake word:** a short phrase that tells the device "I am talking to you", so that it does not act on every sound in the room.

**Python in this lab:** loops, `if`, lists and dictionaries. The helper is `x_voice.py`; it also uses `edge_voice.py` of Lab 21.

In [ ]:
import sys, os, time
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import torch
import pandas as pd
import x_voice as xv
import edge_voice as ev
from IPython.display import Audio, display
from slm_lab import check

## X4.1 The chain on paper
One turn of the conversation has **four stages**. Each one has its own model or code, its own time and its own way to fail.

| Stage | What happens | Made of |
|---|---|---|
| 1. Speech | the recording becomes text | Whisper small |
| 2. Understand | the text becomes an intent and details | the small language model (the reader) |
| 3. Decide and act | the code tracks the task, asks for missing details, checks, changes the device | plain code |
| 4. Speak | the reply text becomes a recording | text-to-speech model |

*In one sentence: the user waits for all four stages, one after the other.*

**Predict.** Write on paper which stage you think takes the **longest**, and which the **shortest**, and a guess in milliseconds for the whole turn.

Now we start the dialogue manager. The word `"auto"` means: the TV manager of X3 with the small model if it works, and the home manager of Day 2 if it does not. To choose by hand write `"tv_model"`, `"tv_rules"`, `"home"` or `"home_rules"`.

In [ ]:
assistant, KIND, WHY = xv.start_assistant("auto")           # 🔧 try "tv_rules" (keyword rules, no model)
CONVERSATIONS = xv.conversations(KIND)
total_turns = 0
for name, sentences in CONVERSATIONS:
    total_turns = total_turns + len(sentences)
print("dialogue manager:", KIND, "-", WHY)
print(len(CONVERSATIONS), "conversations,", total_turns, "turns")

In [ ]:
check(assistant is not None and total_turns > 0, "Checkpoint: the dialogue manager is ready (" + KIND + ").",
      "The dialogue manager did not start. Ask the trainer; try xv.start_assistant('home') to use the Day 2 assistant.")

**You should see** which manager runs and why. Every cell below works with either one: the lab never looks inside the manager. It only calls three functions (`new_assistant`, `reset_assistant`, `take_turn`) that sit in one place in `x_voice.py`. **That is a design rule:** keep the part that may change behind a small door.

## X4.2 The ears and the mouth
We load Whisper (the ears) and the text-to-speech model (the mouth). Then we do one call of each, because **the first call is always slower** (the GPU prepares itself) and would spoil our timing.

In [ ]:
started = time.time()
listener = xv.load_listener()
speaker = xv.load_speaker()
print("Whisper and the speaker loaded in", round(time.time() - started), "seconds; the speaker runs on", speaker["device"])

In [ ]:
audio, rate = xv.speak(speaker, "turn on the light")        # warm-up: one call of each stage
heard, seconds = xv.listen(listener, audio, rate)
_ = xv.take_turn(assistant, "hello")
xv.reset_assistant(assistant)
print("warm-up done. Whisper heard:", repr(heard))

In [ ]:
check(len(heard.split()) >= 2, "Checkpoint: the mouth spoke and the ears heard words.",
      "Whisper heard nothing. Run the loading cell again; if it still fails, ask the trainer.")

## X4.3 The mouth speaks, the ears listen
First the **mouth**: a sentence from the first conversation is turned into audio. You can play it.

In [ ]:
sentence = CONVERSATIONS[0][1][0]
audio, rate = xv.speak(speaker, sentence)
print("sentence:", sentence)
print("audio:", round(len(audio) / rate, 1), "seconds at", rate, "samples per second")
display(Audio(audio, rate=rate))

Now the **ears**. We save the audio as a file, read the file back and let Whisper listen. This is the way a **recorded** `.wav` file would go in. Two details help Whisper a lot: every clip has **0.4 seconds of silence** at both ends (a clip that starts and stops abruptly is often misheard), and we tell Whisper that the language is English.

In [ ]:
path = str(xv.GENERATED_FOLDER / "x4_first.wav")
xv.save_wav(path, audio, rate)
file_audio, file_rate = xv.read_wav(path)
heard, seconds = xv.listen(listener, file_audio, file_rate)
print("said :", sentence)
print("heard:", heard, "(" + str(round(1000 * seconds)) + " ms)")
print("word error rate:", xv.wer(sentence, heard), "%")

**You should see** that Whisper hears the sentence back, with capital letters and a full stop that we did not speak, and perhaps digits instead of words ("23" for "twenty three"). That is why the word error rate compares **cleaned** words: no capitals, no punctuation, numbers as digits.

## X4.4 One full turn, with the answer in audio
We write **one function for the whole chain** so that you can read it from top to bottom: listen, take the turn, speak the reply, and note the time of each stage.

In [ ]:
def voice_turn(clip, say_reply=True):
    heard, seconds = xv.listen(listener, clip["audio"], clip["rate"])
    turn = xv.take_turn(assistant, heard)
    turn["heard"] = heard
    turn["speech_ms"] = round(1000 * seconds)
    turn["speak_ms"] = 0
    if say_reply:
        spoken_at = time.time()
        turn["reply_audio"], turn["reply_rate"] = xv.speak(speaker, xv.speakable(turn["reply"]), pad=False)
        turn["speak_ms"] = round(1000 * (time.time() - spoken_at))
    turn["total_ms"] = turn["speech_ms"] + turn["understand_ms"] + turn["act_ms"] + turn["speak_ms"]
    return turn

The reply text is first made **speakable**: "22 C" becomes "twenty two degrees", symbols are removed and long answers are cut. Then we make the clips of all the conversations: spoken by the model, or your recordings if you put them in `data/voice`.

In [ ]:
CLIPS = xv.make_clips(speaker, CONVERSATIONS)
for c in range(len(CONVERSATIONS)):
    sources = []
    for clip in CLIPS[c]:
        sources.append(clip["source"])
    print(CONVERSATIONS[c][0].ljust(24), sources)

Now one conversation, turn by turn. Play the user's clip and the assistant's answer.

In [ ]:
FIRST = 1
xv.reset_assistant(assistant)
for t in range(len(CONVERSATIONS[FIRST][1])):
    clip = CLIPS[FIRST][t]
    turn = voice_turn(clip)
    print("you said   :", clip["said"], "| heard:", turn["heard"])
    print("it replied :", turn["reply"], "| changed:", turn["changed"])
    print("path       :", " > ".join(turn["path"]), "|", turn["total_ms"], "ms in all")
    display(Audio(turn["reply_audio"], rate=turn["reply_rate"]))

In [ ]:
check(turn["reply"] != "" and "reply_audio" in turn, "Checkpoint: the assistant answered in audio.",
      "No reply was spoken. Run the cell that defines voice_turn, then the clips cell, then this one.")

**You should see** the user's words as Whisper heard them, the reply text, what changed on the device (a plain dictionary: only the keys that this turn changed), the nodes of the graph that ran, and an audio player with the spoken answer. Listen to it. Does it sound right? Are numbers and short forms read well?

## X4.5 All the conversations, and the waterfall
Now we run every conversation from audio and keep one row per turn: the time of the four stages, in order. This table is the **waterfall**.

In [ ]:
waterfall = []
for c in range(len(CONVERSATIONS)):
    xv.reset_assistant(assistant)
    for t in range(len(CONVERSATIONS[c][1])):
        turn = voice_turn(CLIPS[c][t])
        waterfall.append({"conversation": CONVERSATIONS[c][0], "turn": t + 1, "clip": CLIPS[c][t]["source"], "heard": turn["heard"],
                          "reply": turn["reply"], "speech ms": turn["speech_ms"], "understand ms": turn["understand_ms"],
                          "act ms": turn["act_ms"], "speak ms": turn["speak_ms"], "total ms": turn["total_ms"]})
pd.DataFrame(waterfall)

In [ ]:
check(len(waterfall) == total_turns, "Checkpoint: every turn went through the four stages (" + str(len(waterfall)) + " turns).",
      "Some turns are missing. Run the cells of X4.4 again, in order, then this one.")

**You should see** one row per turn. Look at the column `heard` against what was said, and at `reply`. Where is the assistant asking for a missing detail, and where does it act at once? Also look at the four time columns: which is the biggest in most rows?

Now the average of each stage, drawn as a text bar so that you see the shape.

In [ ]:
stages = ["speech ms", "understand ms", "act ms", "speak ms"]
average = {}
whole = 0
for stage in stages:
    total = 0
    for row in waterfall:
        total = total + row[stage]
    average[stage] = round(total / len(waterfall))
    whole = whole + average[stage]
for stage in stages:
    print(stage.ljust(14), str(average[stage]).rjust(5), "ms", "#" * max(1, round(40 * average[stage] / max(whole, 1))))
print("whole turn".ljust(14), str(whole).rjust(5), "ms")

**You should see** where the time goes. Compare with your guess on paper. A few things to remember when you read it:
- The stage *decide and act* is plain code, so it is usually the smallest. The model is not where **all** the delay is.
- The *speak* stage here makes the whole audio before it plays. A real product starts playing after the first words are ready, so the user hears the answer sooner than this table says.
- With the small model on a GPU each stage is a fraction of a second; on a small device with no GPU every stage becomes slower, and the sum is what the user feels.

**Decision.** Write one sentence: if you had one week to make the assistant faster, which stage would you work on first, and what would you try?

## X4.6 The noise experiment
So far the clips were clean. A real room has a fan, a street, other people. We take the **same clips** and add white noise at three levels, then run every conversation again.

For each level we compare with a **reference run** where the sentences go in as clean text. The difference between the noisy run and the reference run is what the **speech stage** costs us.

In [ ]:
NOISE_LEVELS = [("light", 20), ("medium", 12), ("heavy", 6)]        # 🔧 signal-to-noise ratio in dB
reference = xv.run_conversations(assistant, CONVERSATIONS)           # clean text, no speech at all
print("reference run:", len(reference), "turns")
for index in [0, 1]:
    print("said:", reference[index]["said"], "| the manager answered:", reference[index]["reply"])

**Predict.** Write on paper, for the light, medium and heavy noise: how many of the conversations do you think still **end in the right device state**, out of the total? And will the mean word error rate rise **faster** than the number of wrong results, or slower? Why?

The next cell runs the clean speech and the three noisy runs. It takes a little while: every turn is heard, scored and understood again.

In [ ]:
runs = {"clean speech": xv.run_conversations(assistant, CONVERSATIONS, CLIPS, listener)}
for level, snr in NOISE_LEVELS:
    key = level + " (" + str(snr) + " dB)"
    runs[key] = xv.run_conversations(assistant, CONVERSATIONS, CLIPS, listener, snr_db=snr)
print("runs done:", list(runs))

In [ ]:
summary = []
for key in runs:
    numbers = xv.compare_runs(runs[key], reference)
    numbers["run"] = key
    summary.append(numbers)
pd.DataFrame(summary).set_index("run")

In [ ]:
check(len(summary) == 1 + len(NOISE_LEVELS), "Checkpoint: the clean run and " + str(len(NOISE_LEVELS)) + " noisy runs are compared with the text run.",
      "Run the cell that makes the runs again; it needs the listener, the clips and the reference run.")

**You should see** the mean word error rate going up as the noise grows, and the share of turns with the same intent and the same result going down. Look at **two gaps**:
- the gap between *heard exactly* and *same result*: words can be wrong without the result being wrong, because the manager needs only a few key words ("kitchen", "light", "on");
- the gap between *same result* and *conversations ending right*: a conversation has several turns, and **one wrong turn can change the rest**, because the manager keeps the context.

In our test run (the trainer's voice, Whisper small, the TV manager with the model): with clean speech the mean word error rate was 2.2 per cent and all five conversations ended right. At 20 dB it was 6.7 per cent (5 of 5 right), at 12 dB 20.7 per cent (4 of 5), and at 6 dB 44.8 per cent (1 of 5). The result held up longer than the words did.

Compare with your prediction. At which level does the assistant stop being usable for you? A product team chooses this level from the places where the device will stand: a quiet bedroom is not a kitchen with a mixer.

## X4.7 Where does the chain break?
We now sort every noisy turn into four groups, so that we see **which stage broke it**.

In [ ]:
labels = ["all right", "speech wrong, result still right", "speech wrong, result wrong", "speech right, result differs"]
counts = []
for key in runs:
    line = {"run": key}
    for label in labels:
        line[label] = 0
    for index in range(len(reference)):
        label = xv.classify_row(runs[key][index], reference[index])
        line[label] = line[label] + 1
    counts.append(line)
pd.DataFrame(counts).set_index("run")

**You should see** that the group "speech wrong, result wrong" grows with the noise, that some speech mistakes did not matter, and perhaps a few turns where the speech was right but the result was different: that is an **earlier mistake carried over** by the context. Here is one real example from the heaviest run.

In [ ]:
heaviest = list(runs)[len(runs) - 1]
shown = 0
for index in range(len(reference)):
    row = runs[heaviest][index]
    if shown < 2 and not xv.result_same(row, reference[index]):
        print("said     :", row["said"])
        print("heard    :", row["heard"])
        print("reference:", reference[index]["reply"])
        print("it said  :", row["reply"])
        print("---")
        shown = shown + 1

Read the examples. Which stage went wrong first? Now the list of **breaks of the chain** and what you would do about each. Tick on paper the ones you saw in your own tables.

| Where it breaks | What it looks like | What you could do |
|---|---|---|
| The room | noise, echo, a speaker far from the microphone | a better microphone, noise suppression before Whisper, a wake word to start listening |
| Speech recognition | a key word is wrong ("close" heard as "hose"), numbers and names | give the recogniser the list of words the device knows, correct near-misses in code (the nearest known word), ask back when it is unsure |
| Silence and noise | the recogniser writes words that nobody said | a voice-activity check before Whisper, ignore very short or very low-confidence text |
| Understanding | the words are right, the meaning is wrong | the confidence of Lab 20, more training examples, ask back |
| The conversation | one wrong turn changes the context of the next | confirm risky actions, allow "no, I meant ...", forget old context after a pause |
| Speaking | long answers, numbers read badly, the device hears its own voice | short replies, numbers written as words, switch the microphone off while it speaks |
| Time | each stage adds up | start playing the answer early, a smaller model, a smaller prompt, no model for easy commands |

*In one sentence: every stage can fail, and the cheapest place to catch a failure is the earliest stage that can notice it.*

## X4.8 Your turn 1: "did you say ...?"
The speech recogniser knows when it is unsure. We feed its own text back to it and read the **probability of each word piece**; the average is the **speech confidence**. This is the same idea as the intent confidence of Lab 20, one stage earlier. If the confidence is low, the assistant should not act: it should **ask**.

The rows below hold, for every noisy turn, what Whisper heard, its confidence, and whether the final result was right.

In [ ]:
ALL_TURNS = []
for key in runs:
    if key != "clean speech":
        for index in range(len(reference)):
            row = runs[key][index]
            ALL_TURNS.append({"run": key, "heard": row["heard"], "confidence": row["confidence_speech"],
                              "result_right": xv.result_same(row, reference[index])})
known = 0
for item in ALL_TURNS:
    if item["confidence"] is not None:
        known = known + 1
print(len(ALL_TURNS), "noisy turns,", known, "with a speech confidence")

In [ ]:
check(known > 0, "Checkpoint: the speech confidence is known for the noisy turns.",
      "No confidence was computed. Look at xv.listen_confidence(...)['error'] for the reason and ask the trainer.")

Write the function `decide`. It gets the text that was heard and the confidence, and returns **either** the word `ACT` **or** a question that starts with `Did you say`. A wrong action that we avoided is a good question; a question about a turn that was right is a needless one. Choose the sentence of the question yourself.

If you are stuck, open the solution cell below the starter. For the TV manager there is a second signal: `turn["confidence"]` from `take_turn` is the intent confidence of Lab 20. You can combine the two: ask if **either** is low.

In [ ]:
def decide(heard, confidence, threshold):
    # TODO: if the confidence is below the threshold, return a question: Did you say "..."?
    # TODO: otherwise return the word ACT
    return "ACT"

Now we score the rule on all the noisy turns, for several thresholds. For each threshold: how many **useful** questions (the result would have been wrong), how many **needless** ones (the result was right), and how many turns where the assistant **acted although the result was wrong**.

In [ ]:
def score_rule(threshold):
    useful, needless, acted_wrong = 0, 0, 0
    for item in ALL_TURNS:
        if item["confidence"] is None:
            continue
        asked = decide(item["heard"], item["confidence"], threshold) != "ACT"
        if asked and not item["result_right"]:
            useful = useful + 1
        elif asked:
            needless = needless + 1
        elif not item["result_right"]:
            acted_wrong = acted_wrong + 1
    return {"threshold": threshold, "useful questions": useful, "needless questions": needless, "acted although wrong": acted_wrong}

In [ ]:
sweep = []
for threshold in [0.5, 0.7, 0.8, 0.9, 0.95]:
    sweep.append(score_rule(threshold))
pd.DataFrame(sweep)

In [ ]:
check(sweep[2]["useful questions"] + sweep[2]["needless questions"] > 0, "Checkpoint: your rule asks at least one question at 0.8.",
      "The rule never asked. Check that decide returns a question (not ACT) when the confidence is below the threshold.")

**You should see** the trade of Lab 20 again: a higher threshold catches more wrong turns, and asks more needless questions. The table shows where a good threshold is for this noise; there is no free lunch. **Write one sentence:** which threshold would you ship, and for which actions would you always ask, whatever the confidence (think of the risky action in your conversations)?

## X4.9 Your turn 2: a wake word
A device in a living room hears television, talk and traffic all day. It must act **only when it is called**. A **wake word** such as "hey computer" does that. In a product a tiny model that costs almost no power listens all the time for this one phrase, and only then Whisper starts. Here we only check the text that Whisper heard.

We speak four sentences: two with the wake word and two without it.

In [ ]:
first = CONVERSATIONS[0][1][0]
second = CONVERSATIONS[1][1][0]
WAKE_TESTS = [("hey computer " + first, True), (first, False), ("hey computer " + second, True), ("we should go to the market now", False)]
HEARD_WAKE = []
for sentence, has_wake in WAKE_TESTS:
    audio, rate = xv.speak(speaker, sentence)
    HEARD_WAKE.append(xv.listen(listener, audio, rate)[0])
print(HEARD_WAKE)

Write `has_wake_word(heard)`: it returns `True` only if the text starts with the wake word. Whisper may write "Hey, computer." or split a word, so do not demand an exact match. The helper `xv.clean_words(text)` gives a list of lower-case words without punctuation, and `xv.edit_distance(first, second)` tells how many letters differ between two texts (0 means the same).

In [ ]:
def has_wake_word(heard):
    words = xv.clean_words(heard)          # for example ["hey", "computer", "turn", "on"]
    # TODO: return True only if the first two words together are close to "heycomputer" (allow a few letters of difference)
    return True

In [ ]:
correct = 0
for index in range(len(WAKE_TESTS)):
    answer = has_wake_word(HEARD_WAKE[index])
    print(str(answer).ljust(6), "expected", str(WAKE_TESTS[index][1]).ljust(6), HEARD_WAKE[index])
    if answer == WAKE_TESTS[index][1]:
        correct = correct + 1
print("correct:", correct, "of", len(WAKE_TESTS))

In [ ]:
check(correct == len(WAKE_TESTS), "Checkpoint: the wake word check is right on all four sentences.",
      "Print the words Whisper heard and look at the first two. Is the distance to heycomputer small for the ones with the wake word?")

**You should see** the rule accept the two sentences with the wake word and refuse the other two. **Think:** what happens if the distance limit is too small (the device does not wake when it should), or too big (it wakes at any sound)? That is the same trade as the confidence threshold: it decides who is annoyed, the user who is ignored or the user who is woken at night.

## What you take away
- A voice assistant is a **chain of four stages**. Measure each stage in milliseconds (the waterfall), because the user waits for the sum and only the biggest stage is worth your first week.
- **Noise hurts the speech stage first.** Some speech mistakes do not matter because the manager needs only a few key words; others change the result, and a wrong turn can carry over into the next turns through the context.
- Put a cheap check at the **earliest stage that can notice** a problem: a wake word, the speech confidence, then the intent confidence of Lab 20, and always confirm risky actions.

## Check your understanding
1. Why do we draw the waterfall for each stage instead of measuring only the total time?
*✍️ Your answer:* 
2. In the noisy runs some turns had a wrong word and still gave the right result. Why, and is that a reason not to worry about the speech stage?
*✍️ Your answer:* 
3. The assistant misheard one turn at the start of a conversation, and the later turns, which were heard correctly, also gave different results. Why?
*✍️ Your answer:* 
4. You set the speech-confidence threshold very high. What happens to the wrong actions and to the user?
*✍️ Your answer:* 

## Next
This is the last of the four extra labs. The bridge is your own product: take the chain of this lab, put your own device behind the dialogue manager, record your own sentences in `data/voice` in the places where the device will stand, and repeat the noise experiment with the noise of that room.